In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')


In [2]:
#df = pd.read_csv('steam_indie_merged_data.csv', engine='python')
df = pd.read_csv('../../../data/steam_indie_9692.csv')

In [3]:
df.T

,0,1,2,3,4,5,6,7,8,9,...,9682,9683,9684,9685,9686,9687,9688,9689,9690,9691
appid,899770,251570,1116170,1326470,2186680,526870,2881650,513710,1144200,1145350,...,1504090,2490750,2396510,2289670,2260720,2607200,541910,2429330,2218570,2656520
name,Last Epoch,7 Days to Die,CyberCorp,Sons Of The Forest,"Warhammer 40,000: Rogue Trader",Satisfactory,Content Warning,SCUM,Ready or Not,Hades II,...,Ascendum,The Dungeon Tower,Castaway Station,Ghosting Vandal,Fast Burger Simulator,ONE BUTTON RUN,David Slade Mysteries: Case Files,VR-太阳系,Idle interstellar Factory 2,Mycelium Heaven
owners,"20,000,000 .. 50,000,000","10,000,000 .. 20,000,000","10,000,000 .. 20,000,000","10,000,000 .. 20,000,000","10,000,000 .. 20,000,000","10,000,000 .. 20,000,000","5,000,000 .. 10,000,000","5,000,000 .. 10,000,000","5,000,000 .. 10,000,000","2,000,000 .. 5,000,000",...,"0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000","0 .. 20,000"
positive,88027,327889,266,222495,26360,225479,137483,83704,210654,60844,...,18,14,14,14,14,12,12,11,10,33
negative,22596,42157,56,31051,4445,6585,7608,28411,25542,3288,...,0,0,2,1,5,0,2,3,3,3
price,3499,4499,1499,2999,4999,3999,799,1799,4999,2399,...,1399,499,1599,999,299,299,999,199,499,999
ccu,5831,17045,3,4450,3582,12596,722,8535,4296,1838,...,0,0,0,0,0,0,0,0,0,0
type,game,game,game,game,game,game,game,game,game,game,...,game,game,game,game,game,game,game,game,game,game
genres,"['Action', 'Adventure', 'Indie', 'RPG']","['Action', 'Adventure', 'Indie', 'RPG', 'Simul...","['Action', 'Adventure', 'Indie', 'RPG']","['Action', 'Adventure', 'Indie', 'Simulation']","['Action', 'Adventure', 'Indie', 'RPG', 'Strat...","['Adventure', 'Indie', 'Simulation', 'Strategy']","['Action', 'Adventure', 'Indie']","['Action', 'Adventure', 'Indie', 'Massively Mu...","['Action', 'Adventure', 'Indie']","['Action', 'Indie', 'RPG']",...,"['Adventure', 'Casual', 'Indie']","['Action', 'Adventure', 'Casual', 'Indie', 'RPG']","['Adventure', 'Indie', 'Strategy']","['Action', 'Adventure', 'Indie']","['Adventure', 'Casual', 'Indie', 'Simulation']","['Adventure', 'Casual', 'Indie']","['Action', 'Adventure', 'Indie']","['Casual', 'Indie', 'Simulation']","['Indie', 'Simulation', 'Strategy']","['Casual', 'Indie']"
release_date,2024-02-21,2024-07-25,2025-04-22,2024-02-22,2023-12-07,2024-09-10,2024-04-01,2025-06-17,2023-12-13,2025-09-25,...,2023-04-12,2023-09-11,2023-07-12,2025-09-19,2023-01-09,2023-10-19,2023-01-25,2023-05-27,2023-02-08,2025-02-27


In [4]:
import pandas as pd
import numpy as np
import ast

# 1) 파일 불러오기
df = pd.read_csv('../../../data/steam_indie_9692.csv')

print(df.head())
print(df.shape)
print(df.columns.tolist())


# -----------------------------
# 2) 기본 복사
# -----------------------------
df_clean = df.copy()


# -----------------------------
# 3) 문자열 컬럼 정리
# -----------------------------
text_cols = ["spy_name", "name_store", "owners", "type", "genres", "release_date", "developers"]

for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype("string").str.strip()


# -----------------------------
# 4) owners 분해
# 예: "50,000,000 .. 100,000,000"
# -----------------------------
df_clean["owners"] = df_clean["owners"].str.replace(",", "", regex=False)

df_clean["owners_low"] = df_clean["owners"].str.split(r"\.\.").str[0].str.strip()
df_clean["owners_high"] = df_clean["owners"].str.split(r"\.\.").str[1].str.strip()

df_clean["owners_low"] = pd.to_numeric(df_clean["owners_low"], errors="coerce")
df_clean["owners_high"] = pd.to_numeric(df_clean["owners_high"], errors="coerce")

df_clean["owners_mid"] = (df_clean["owners_low"] + df_clean["owners_high"]) / 2


# -----------------------------
# 5) 수치형 컬럼 변환
# -----------------------------
numeric_cols = ["appid", "positive", "negative", "price_spy", "ccu"]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")


# -----------------------------
# 6) 가격 달러 단위로 변환
# 예: 2999 -> 29.99
# -----------------------------
df_clean["price"] = df_clean["price"] / 100


# -----------------------------
# 7) 리뷰 파생변수
# -----------------------------
df_clean["review_total"] = df_clean["positive"].fillna(0) + df_clean["negative"].fillna(0)

df_clean["positive_ratio"] = np.where(
    df_clean["review_total"] > 0,
    df_clean["positive"] / df_clean["review_total"],
    np.nan
)

df_clean["negative_ratio"] = np.where(
    df_clean["review_total"] > 0,
    df_clean["negative"] / df_clean["review_total"],
    np.nan
)

df_clean["log_review_total"] = np.log1p(df_clean["review_total"])


# -----------------------------
# 8) genres 문자열 -> 실제 리스트
# -----------------------------
def parse_genres(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

df_clean["genres_list"] = df_clean["genres"].apply(parse_genres)


# -----------------------------
# 9) 장르 플래그 생성
# -----------------------------
def has_genre(genres, genre_name):
    if not isinstance(genres, list):
        return 0
    return int(genre_name in genres)

genre_targets = [
    "Action", "Adventure", "RPG", "Strategy",
    "Simulation", "Casual", "Free To Play", "Early Access", "Indie"
]

for g in genre_targets:
    col_name = "genre_" + g.lower().replace(" ", "_")
    df_clean[col_name] = df_clean["genres_list"].apply(lambda x: has_genre(x, g))


# -----------------------------
# 10) 날짜 처리
# -----------------------------
df_clean["release_date_parsed"] = pd.to_datetime(
    df_clean["release_date"],
    format="%d %b, %Y",
    errors="coerce"
)

today = pd.Timestamp.today().normalize()

df_clean["days_since_release"] = (today - df_clean["release_date_parsed"]).dt.days
df_clean["release_year"] = df_clean["release_date_parsed"].dt.year
df_clean["release_month"] = df_clean["release_date_parsed"].dt.month


# -----------------------------
# 11) 무료 게임 여부
# -----------------------------
df_clean["is_free"] = (df_clean["price"] == 0).astype(int)


# -----------------------------
# 12) 가격 구간화
# -----------------------------
df_clean["price_band"] = pd.cut(
    df_clean["price"],
    bins=[-1, 0, 5, 10, 20, 30, 60, 9999],
    labels=["Free", "0~5", "5~10", "10~20", "20~30", "30~60", "60+"]
)


# -----------------------------
# 13) 리뷰 규모 구간화
# -----------------------------
df_clean["review_band"] = pd.cut(
    df_clean["review_total"],
    bins=[-1, 10, 50, 100, 500, 1000, 10000, 999999999],
    labels=["0~10", "11~50", "51~100", "101~500", "501~1000", "1001~10000", "10000+"]
)


# -----------------------------
# 14) 이름 정리용 대표 이름 만들기
# store 이름 우선, 없으면 spy 이름 사용
# -----------------------------
df_clean["game_name"] = df_clean["name"].fillna(df_clean["name"])


# -----------------------------
# 15) 최종 확인
# -----------------------------
print(df_clean.head())
print(df_clean.shape)
print(df_clean.columns.tolist())
print(df_clean.isna().sum().sort_values(ascending=False).head(20))


# -----------------------------
# 16) 저장
# -----------------------------
df_clean.to_csv("steam_indie_preprocessed.csv", index=False, encoding="utf-8-sig")
print("저장 완료: steam_indie_preprocessed.csv")

     appid                            name                    owners  \
0   899770                      Last Epoch  20,000,000 .. 50,000,000   
1   251570                   7 Days to Die  10,000,000 .. 20,000,000   
2  1116170                       CyberCorp  10,000,000 .. 20,000,000   
3  1326470              Sons Of The Forest  10,000,000 .. 20,000,000   
4  2186680  Warhammer 40,000: Rogue Trader  10,000,000 .. 20,000,000   

   positive  negative  price    ccu  type  \
0     88027     22596   3499   5831  game   
1    327889     42157   4499  17045  game   
2       266        56   1499      3  game   
3    222495     31051   2999   4450  game   
4     26360      4445   4999   3582  game   

                                              genres release_date  \
0            ['Action', 'Adventure', 'Indie', 'RPG']   2024-02-21   
1  ['Action', 'Adventure', 'Indie', 'RPG', 'Simul...   2024-07-25   
2            ['Action', 'Adventure', 'Indie', 'RPG']   2025-04-22   
3     ['Action', 'Ad

In [5]:
print(df_clean[[
    "price",
    "review_total",
    "positive_ratio",
    "owners_mid",
    "days_since_release"
]].describe())

             price   review_total  positive_ratio    owners_mid  \
count  9692.000000    9692.000000     9692.000000  9.692000e+03   
mean      8.837071     793.709554        0.841811  5.871801e+04   
std      10.585507    7662.793747        0.152017  5.358492e+05   
min       0.000000      10.000000        0.000000  1.000000e+04   
25%       2.990000      18.000000        0.770653  1.000000e+04   
50%       5.990000      41.000000        0.883721  1.000000e+04   
75%      11.990000     152.000000        0.953416  3.500000e+04   
max     199.990000  370046.000000        1.000000  3.500000e+07   

       days_since_release  
count                 0.0  
mean                  NaN  
std                   NaN  
min                   NaN  
25%                   NaN  
50%                   NaN  
75%                   NaN  
max                   NaN  


In [6]:
#가격별 리뷰수
print(
    df_clean.groupby("price_band", observed=False)["review_total"]
    .mean()
    .sort_values(ascending=False)
)

price_band
30~60    15253.325301
20~30     3507.120000
10~20     1258.216867
Free       677.338542
5~10       479.571588
0~5        253.674352
60+         20.750000
Name: review_total, dtype: float64


In [7]:
#장르별 리뷰량
genre_cols = [
    "genre_action", "genre_adventure", "genre_rpg",
    "genre_strategy", "genre_simulation", "genre_casual"
]

for col in genre_cols:
    print(f"\n[{col}]")
    print(df_clean.groupby(col)["review_total"].mean())


[genre_action]
genre_action
0     575.420075
1    1092.317615
Name: review_total, dtype: float64

[genre_adventure]
genre_adventure
0    599.837257
1    990.727689
Name: review_total, dtype: float64

[genre_rpg]
genre_rpg
0     644.539477
1    1303.498633
Name: review_total, dtype: float64

[genre_strategy]
genre_strategy
0     714.236763
1    1098.401496
Name: review_total, dtype: float64

[genre_simulation]
genre_simulation
0     603.915287
1    1338.827807
Name: review_total, dtype: float64

[genre_casual]
genre_casual
0    1018.64630
1     488.34055
Name: review_total, dtype: float64


In [8]:
print(df_clean.shape)
print('-'*50)
print(df_clean.columns.tolist())
print('-'*50)
print(df_clean.head())
print('-'*50)
print(df_clean.info())

(9692, 40)
--------------------------------------------------
['appid', 'name', 'owners', 'positive', 'negative', 'price', 'ccu', 'type', 'genres', 'release_date', 'developers', 'total_reviews', 'owners_lower', 'is_f2p', 'is_early_access', 'owners_low', 'owners_high', 'owners_mid', 'review_total', 'positive_ratio', 'negative_ratio', 'log_review_total', 'genres_list', 'genre_action', 'genre_adventure', 'genre_rpg', 'genre_strategy', 'genre_simulation', 'genre_casual', 'genre_free_to_play', 'genre_early_access', 'genre_indie', 'release_date_parsed', 'days_since_release', 'release_year', 'release_month', 'is_free', 'price_band', 'review_band', 'game_name']
--------------------------------------------------
     appid                            name                owners  positive  \
0   899770                      Last Epoch  20000000 .. 50000000     88027   
1   251570                   7 Days to Die  10000000 .. 20000000    327889   
2  1116170                       CyberCorp  10000000 

In [9]:
na_summary = df_clean.isna().sum().sort_values(ascending=False)
print(na_summary.head(20))

release_month          9692
release_year           9692
days_since_release     9692
release_date_parsed    9692
developers               12
appid                     0
owners                    0
name                      0
type                      0
ccu                       0
price                     0
negative                  0
positive                  0
genres                    0
release_date              0
total_reviews             0
owners_high               0
owners_mid                0
review_total              0
positive_ratio            0
dtype: int64


In [10]:
df_clean['positive'].isna().sum()

np.int64(0)

In [11]:
df_clean['positive_ratio'].isna().sum()

np.int64(0)

In [12]:
df_clean['negative'].isna().sum()

np.int64(0)

In [13]:
df_clean['negative_ratio'].isna().sum()

np.int64(0)

In [14]:
df_clean.sort_values(by='review_total', ascending=False).head()

,appid,name,owners,positive,negative,price,ccu,type,genres,release_date,...,genre_early_access,genre_indie,release_date_parsed,days_since_release,release_year,release_month,is_free,price_band,review_band,game_name
1,251570,7 Days to Die,10000000 .. 20000000,327889,42157,44.99,17045,game,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,...,0,1,NaT,NaN,NaN,NaN,0,30~60,10000+,7 Days to Die
3,1326470,Sons Of The Forest,10000000 .. 20000000,222495,31051,29.99,4450,game,"['Action', 'Adventure', 'Indie', 'Simulation']",2024-02-22,...,0,1,NaT,NaN,NaN,NaN,0,20~30,10000+,Sons Of The Forest
8,1144200,Ready or Not,5000000 .. 10000000,210654,25542,49.99,4296,game,"['Action', 'Adventure', 'Indie']",2023-12-13,...,0,1,NaT,NaN,NaN,NaN,0,30~60,10000+,Ready or Not
5,526870,Satisfactory,10000000 .. 20000000,225479,6585,39.99,12596,game,"['Adventure', 'Indie', 'Simulation', 'Strategy']",2024-09-10,...,0,1,NaT,NaN,NaN,NaN,0,30~60,10000+,Satisfactory
16,1468810,鬼谷八荒 Tale of Immortal,2000000 .. 5000000,121020,106084,12.99,4116,game,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2023-05-26,...,0,1,NaT,NaN,NaN,NaN,0,10~20,10000+,鬼谷八荒 Tale of Immortal


In [15]:
df_clean[df_clean['review_total'] > 30].sort_values(by='review_total', ascending=True).head()

,appid,name,owners,positive,negative,price,ccu,type,genres,release_date,...,genre_early_access,genre_indie,release_date_parsed,days_since_release,release_year,release_month,is_free,price_band,review_band,game_name
7507,2396090,Aery - Flow of Time,0 .. 20000,29,2,9.99,0,game,"['Adventure', 'Casual', 'Indie', 'Simulation']",2023-08-18,...,0,1,NaT,NaN,NaN,NaN,0,5~10,11~50,Aery - Flow of Time
4825,2525830,The Lost Prince,0 .. 20000,27,4,12.99,0,game,"['Indie', 'Simulation']",2023-12-11,...,0,1,NaT,NaN,NaN,NaN,0,10~20,11~50,The Lost Prince
6369,3041200,My Party Needs An Alchemist,0 .. 20000,30,1,19.99,0,game,"['Adventure', 'Casual', 'Indie', 'Simulation']",2025-03-27,...,0,1,NaT,NaN,NaN,NaN,0,10~20,11~50,My Party Needs An Alchemist
1918,3316640,Neon Hearts City,20000 .. 50000,26,5,9.99,0,game,"['Adventure', 'Indie']",2025-05-13,...,0,1,NaT,NaN,NaN,NaN,0,5~10,11~50,Neon Hearts City
6370,3214390,Stranger Watch,0 .. 20000,30,1,5.99,4,game,"['Action', 'Adventure', 'Indie']",2024-11-29,...,0,1,NaT,NaN,NaN,NaN,0,5~10,11~50,Stranger Watch


In [16]:
df_clean["genres_list"].dtype

dtype('O')

In [17]:
all_genres = df_clean["genres_list"].explode().dropna()

print(all_genres.unique())
print("장르 개수:", len(all_genres.unique()))

<StringArray>
[               'Action',             'Adventure',                 'Indie',
                   'RPG',            'Simulation',              'Strategy',
 'Massively Multiplayer',                'Casual',                'Racing',
                'Sports',             'Utilities',            'Accounting',
      'Video Production', 'Design & Illustration',         'Photo Editing',
             'Education',     'Software Training',  'Animation & Modeling',
      'Game Development',        'Web Publishing',      'Audio Production']
Length: 21, dtype: str
장르 개수: 21


In [18]:
genre_counts = all_genres.value_counts()

genre_counts

genres_list
Indie                    9687
Adventure                4807
Casual                   4111
Action                   4093
Simulation               2503
RPG                      2194
Strategy                 2005
Sports                    341
Racing                    301
Massively Multiplayer     124
Utilities                  21
Design & Illustration      10
Animation & Modeling        7
Education                   6
Software Training           6
Game Development            5
Video Production            4
Audio Production            4
Photo Editing               2
Web Publishing              2
Accounting                  1
Name: count, dtype: int64

In [19]:
df_2024_after = df_clean[df_clean["release_date_parsed"] >= "2024-01-01"].copy()
print("-"*50)
print(df_2024_after.shape)
print("-"*50)
print(df_2024_after.head())
print("-"*50)
print(df_2024_after.tail())

--------------------------------------------------
(0, 40)
--------------------------------------------------
Empty DataFrame
Columns: [appid, name, owners, positive, negative, price, ccu, type, genres, release_date, developers, total_reviews, owners_lower, is_f2p, is_early_access, owners_low, owners_high, owners_mid, review_total, positive_ratio, negative_ratio, log_review_total, genres_list, genre_action, genre_adventure, genre_rpg, genre_strategy, genre_simulation, genre_casual, genre_free_to_play, genre_early_access, genre_indie, release_date_parsed, days_since_release, release_year, release_month, is_free, price_band, review_band, game_name]
Index: []

[0 rows x 40 columns]
--------------------------------------------------
Empty DataFrame
Columns: [appid, name, owners, positive, negative, price, ccu, type, genres, release_date, developers, total_reviews, owners_lower, is_f2p, is_early_access, owners_low, owners_high, owners_mid, review_total, positive_ratio, negative_ratio, log_r

In [20]:
genre_counts = all_genres.value_counts()

genre_counts

genres_list
Indie                    9687
Adventure                4807
Casual                   4111
Action                   4093
Simulation               2503
RPG                      2194
Strategy                 2005
Sports                    341
Racing                    301
Massively Multiplayer     124
Utilities                  21
Design & Illustration      10
Animation & Modeling        7
Education                   6
Software Training           6
Game Development            5
Video Production            4
Audio Production            4
Photo Editing               2
Web Publishing              2
Accounting                  1
Name: count, dtype: int64

In [ ]:
#df.to_csv("preprocesing(1).csv")